In [36]:
with open(r"E:\training\logger\text.txt") as f:
    text = f.read()
chunk = []
start = 0

while start<len(text):
    end = start + 200
    chunk.append(text[start:end])
    start += 200 - 50

print(len(chunk))
print(sum(len(c.split()) for c in chunk)/len(chunk))


24
27.416666666666668


In [24]:
import chromadb
from chromadb.config import Settings

class VectorStore:
    def __init__(self,path = ".\vectorstore",collection_name ="dsa_docs"):
        self.client = chromadb.Client(Settings(persist_directory=path,anonymized_telemetry=False))
        self.collection = self.client.get_or_create_collection(name=collection_name)
    
    def add_chunks(self,chunks,embed_fn):
        if self.collection.count()>0:
            print("Loading_existing")
            return
        print("Building new vector store")
        
        embeddings = []
        ids = []
        for idx,chunk in enumerate(chunks):
            embeddings.append(embed_fn(chunk))
            ids.append(str(idx))
        
        self.collection.add(documents=chunks,embeddings=embeddings,ids=ids)
        
        
    def search(self,query_vector,top_k=3):
        result = self.collection.query(query_embeddings=[query_vector],n_results=top_k)
        return result["documents"][0] if result["documents"] else []


In [ ]:
import nltk
import ollama
from nltk.corpus import stopwords
from math import sqrt
nltk.download("stopwords")

class DocumentLoader():
    def __init__(self,vector_store):
        self.chunks = []
        self.embedded_chunks = []
        self.text = ""
        self.vector_store = vector_store
        
    def load_file(self,path):
        with open(path) as f:
            self.text = f.read()
        return self.text
    
    def chunks_creation(self, size=200, overlap=50):
        self.chunks = []
        start = 0
        while start < len(self.text):
            self.chunks.append(self.text[start:start+size])
            start += size - overlap
        return self.chunks
    
    def embed_all_chunks(self,embed_fn):
        self.vector_store.add_chunks(self.chunks,embed_fn)
    
    def top_3_chunks(self,qurey_vector):
        return self.vector_store.search(qurey_vector,top_k=3)
    
    
    #week3 starting 
    # converting chunks into the vectors
    # def embed_all_chunks(self,model = "qwen2.5"):
    #     self.embedded_chunks.clear()
    #     for chunk in self.chunks:
    #         response = ollama.embeddings(model=model,prompt=chunk)
    #         vector =response['embedding']
    #         self.embedded_chunks.append({"text":chunk,"vector":vector})
    
    # def clean_text(self,text):
    #     stop_words = set(stopwords.words("english"))
    #     text = text.lower().split()
    #     return[word for word in text if word not in stop_words ]
    
    # def scoring(self, keywords, chunk):
    #     chunk_words = chunk.lower().split()
    #     score = 0
    #     for k in keywords:
    #         if " " in k:  # phrase
    #             if k in chunk.lower():
    #                 score += 3
    #         else:
    #             score += chunk_words.count(k)

    #     return score
    
    
    # def cosinesimilarity(self,v1,v2):
    #     dot_product = sum(a*b for a,b in zip(v1,v2))
    #     magnitudev1 = sqrt(sum(a*a for a in v1)) 
    #     magnitudev2 = sqrt(sum(a*a for a in v2)) 
    #     if magnitudev1 ==0 or magnitudev2 == 0:
    #         return 0.0
    #     similarity = (dot_product)/(magnitudev1 * magnitudev2)
    #     return similarity

    # def top_3_chunks(self, user_input):
    #     scored_chunks= []       
    #     for item in self.embedded_chunks:
    #         #score = self.scoring(keyword,chunk)
    #         score = self.cosinesimilarity(user_input,item["vector"])
    #         scored_chunks.append({"score":score,"text":item["text"]})
    #     # sort by score (descending)
    #     scored_chunks.sort(key=lambda x: x["score"], reverse=True)
    #     return scored_chunks[:3]

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\25020020\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [26]:
import ollama
from collections import OrderedDict,deque
import re

class ChatMemory:
    # -----------------------------------------------------------------------------------------------------------------------------------
    # 1.Initiation
    # -----------------------------------------------------------------------------------------------------------------------------------
    def __init__(self,model = "qwen2.5",system_prompt = "be a tutor",max_tokens=3000,cache_size=10,path=None):
        self.model = model
        self.max_tokens = max_tokens
        self.cache_size = cache_size
        self.chat_history = deque([{"role":"system","content":system_prompt}])
        self.cache = OrderedDict()
        self.ai_counter = 0
        self.cache_counter = 0
        self.stack = []
        
        self.vector_store = VectorStore(
            path="./vectorstore",
            collection_name="dsa_docs"
        )

        self.loader = DocumentLoader(self.vector_store)
        self.path = path
        self.doc_text = ''
    # -----------------------------------------------------------------------------------------------------------------------------------
    # 2.DOCUMENT PREPARATION
    # -----------------------------------------------------------------------------------------------------------------------------------
    # prepraing the documents by opening and converting them into chuncks    
    
    def prepare_document(self,path,size=200,overlap=50):
        self.doc_text = self.loader.load_file(path)
        self.loader.chunks_creation(size,overlap)
        self.loader.embed_all_chunks(self.embed)
        
    # -----------------------------------------------------------------------------------------------------------------------------------
    # 3.TEXT NORMALIZATION & KEYWORDS
    # -----------------------------------------------------------------------------------------------------------------------------------   
    # user input preprocessing and toaken calculation    
    def normalize(self,text):
        return re.sub(r'[^\w\s]','', text).strip().lower()
         
    #extracting the keywords and remove the stopwords after the input quistions converted into normailzed text
    # def extract_keywords(self,text):
    #     text = self.normalize(text)
    #     return self.loader.clean_text(text)
    
    # -----------------------------------------------------------------------------------------------------------------------------------
    # 4. EMBEDDING
    # -----------------------------------------------------------------------------------------------------------------------------------      

    
    # embedding the user input
    def embed(self,text):
        response = ollama.embeddings(model=self.model,prompt=text)
        return response["embedding"]
    
    # -----------------------------------------------------------------------------------------------------------------------------------
    # 5. CONTEXT RETRIVAL
    # -----------------------------------------------------------------------------------------------------------------------------------      
    #retriving the context where it is availabe in the doc or not
    def retrive_context(self,user_text):
        #keyword = self.extract_keywords(user_text)
        embedded_user = self.embed(user_text)
        top_chunks = self.loader.top_3_chunks(embedded_user)
        if not top_chunks:
            return "No Context Found"
        #return "\n\n".join(f"[score={c['score']}]\n{c['text']}"for c in top_chunks)
        combined = " ".join (top_chunks)
        if len(combined.split())<80:
            return "No Context Found"
        return "\n\n".join(top_chunks)
    
    # -----------------------------------------------------------------------------------------------------------------------------------
    # 6. TOKEN MANAGEMENT
    # -----------------------------------------------------------------------------------------------------------------------------------          
    def count_tokens(self,text):
        return len(text)//4
    
    def total_tokens(self):
        content = " ".join(x["content"] for x in self.chat_history)
        total_token_used = self.count_tokens(content)
        return total_token_used
    
    def trim_to_budget(self):
        while self.total_tokens()>self.max_tokens:
            del self.chat_history[1]
            del self.chat_history[1]
            
    # -----------------------------------------------------------------------------------------------------------------------------------
    # 7. CAHT HISTORY MANAGEMENT
    # -----------------------------------------------------------------------------------------------------------------------------------      
    #adding user input & reply to the history 
    def add_user(self, text):
        text = self.normalize(text)
        self.chat_history.append({"role":"user","content":text})
        self.trim_to_budget()
        return text
    
    def add_assistant(self,text):
        self.chat_history.append({"role":"assistant","content":text})
        
    def get_history(self):
        return list(self.chat_history)
    
    # -----------------------------------------------------------------------------------------------------------------------------------
    # 8.CACHE MANAGEMENT
    # -----------------------------------------------------------------------------------------------------------------------------------          
    def check_cache(self,text):
        if text in self.cache:
            self.cache_counter+=1
            return self.cache[text]
        else: return None
    
    def store_cache(self,text,ai_reply):
        if len(self.cache)>=self.cache_size:
            self.cache.popitem(last=False)
        self.cache[text] = ai_reply
        self.ai_counter+=1
        
    def get_stats(self):
        total = self.cache_counter + self.ai_counter
        if total == 0:
            return "hit rate: 0%"
        return f"hit rate: {(self.cache_counter / total) * 100:.2f}%"
    
    # -----------------------------------------------------------------------------------------------------------------------------------
    # 9. LLM CHAIN
    # -----------------------------------------------------------------------------------------------------------------------------------   
    # prompt chaining    
    def chain(self,user_):
        context = self.retrive_context(user_)
        if context == "No Context Found":
            return "I can only answer the question about the document loaded. That topic is not covered"
        promt = f"""
        You are a tutor.
        IMPORTANT RULES:
        - Answer ONLY using the information present in the DOCUMENT CONTEXT.
        - Do NOT add new knowledge.
        - Do NOT infer beyond the document.
        - If the answer is not explicitly present, reply exactly:
        "Context not available in the provided document."
        DOCUMENT CONTEXT:
        {context}
        USER QUESTION:
        {user_}
        """
        r1 = ollama.chat(model=self.model,messages=[{"role":"user","content":promt}])
        base = r1["message"]["content"]
        self.stack.append(base)
        r2 = ollama.chat(model=self.model,messages=[{"role":"user","content":f"refine this :\n {base}"}])
        refined_ = r2["message"]["content"]
        self.stack.append(refined_)
        if "error" in refined_.lower() or len(refined_.strip())<5:
            self.stack.pop()
            r2 = ollama.chat(model=self.model,messages=[{"role":"user","content":f"refine this :\n {base}"}])
            refined_ = r2["message"]["content"]
        return refined_
    
    # -----------------------------------------------------------------------------------------------------------------------------------
    # 10. MAIN CHAT ENTRY POINT
    # -----------------------------------------------------------------------------------------------------------------------------------   
    # main function to combine all
    def chat(self,user_):
        normalized = self.add_user(user_)
        cached = self.check_cache(normalized)
        if cached:
            reply = cached
        else:
            reply = self.chain(normalized)
            self.store_cache(normalized,reply)
            
        self.add_assistant(reply)
        return reply 

In [27]:
memory = ChatMemory()
memory.prepare_document(
    path=r"E:\training\logger\text.txt",
    size=200,
    overlap=50)
# memory.loader.embed_all_chunks(model='qwen2.5')
while True:
    user = input("chat: ")
    print("user :",user)
    if user.lower() == "quit":
        print(memory.get_stats())
        break
    reply = memory.chat(user)
    print("qwen2.5:", reply)

Loading_existing
user : quit
hit rate: 0%


In [40]:
import math
from math import sqrt
def cosinesimilarity(v1,v2):
    dot_product = sum(a*b for a,b in zip(v1,v2))
    magnitudev1 = sqrt(sum(a*a for a in v1)) 
    magnitudev2 = sqrt(sum(a*a for a in v2)) 
    
    similarity = (dot_product)/(magnitudev1 * magnitudev2)
    
    return similarity


v1 = [1,0,1,0]
v2 = [1,0,0,1]
v3 = [0,1,0,0]

print(cosinesimilarity(v1,v2))
print(cosinesimilarity(v1,v3))

0.4999999999999999
0.0
